# E12: bounded independent data-generation audit

Run from the repository or a subdirectory with the project's Python dependencies installed. No GPU or model training. The raw files and existing v2 validation features are required. All raw field analysis excludes held-out test rows. This is a development audit, not final test evaluation.

The identical code below was executed as temporary scripts on 26 October 2026; these code cells intentionally have no fabricated execution counts. The final markdown contains a snapshot of the actual persisted results. Full narrative and limitations: `docs/DATA_GENERATION_AUDIT.md`.

Stages 1-2 were specified before inspecting their output; the metadata uncertainty, E11 coverage and full metric cross-checks were follow-ups. Intervals are exploratory, not multiplicity-adjusted. Whole-search resampling is the default; key differences also receive a query-cluster sensitivity check.

Generated large artifacts stay in `outputs/data_generation_audit/`, ignored by Git. Stage 5 registers the new fixed ranking signals via the shared evaluation logger; existing E12 model names are skipped to avoid duplicate registry entries.


## 1. Full training/validation scan and independent ranking calculation


In [ ]:
"""Bounded, read-only data audit. Never examines held-out test records' fields."""
from pathlib import Path
import sys, json, time, itertools, hashlib
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'scripts/evaluation.py').exists())
sys.path.insert(0, str(ROOT))
from scripts.evaluation import expected_metrics, validate_evaluation
OUT = ROOT/'outputs/data_generation_audit'
OUT.mkdir(parents=True, exist_ok=True)
t0 = time.time()
def say(s): print(f'[{time.time()-t0:.1f}s] {s}', flush=True)
def save(name, obj):
    (OUT/f'{name}.json').write_text(json.dumps(obj, indent=2, default=lambda x: x.item() if isinstance(x,np.generic) else str(x)))
split = pd.read_csv(ROOT/'data/processed/split_v1.csv', index_col='session_id')
train_ids = set(split.index[split.split.eq('train')])
val_ids = set(split.index[split.split.eq('validation')])
assert not train_ids.intersection(val_ids)
f = pd.read_parquet(ROOT/'data/processed/features_v2_validation.parquet').sort_values(['session_id','wid']).reset_index(drop=True)
sid = f.session_id.to_numpy(); y = f.label.to_numpy(float)
starts = np.r_[0, np.flatnonzero(np.diff(sid))+1]; ends = np.r_[starts[1:],len(f)]
search_ids = sid[starts]; size = ends-starts; sg = np.repeat(np.arange(len(size)),size)
assert set(search_ids)==val_ids
loc = {int(s):(int(a),int(b)) for s,a,b in zip(search_ids,starts,ends)}
items = np.unique(f.wid.to_numpy()); lookup = {str(w):i for i,w in enumerate(items)}
fi = np.searchsorted(items, f.wid.to_numpy())
names=['events','users','age_gt_1d','age_gt_2d','age_gt_7d','click','cart','ord','flw','users_gt_1d']
hist=np.zeros((len(items),len(names)),dtype=np.int64)
exp=np.zeros((5,len(items)),dtype=np.int32); eng=exp.copy()
fold_tot=np.zeros((5,2),dtype=np.int64)
own_extra=np.zeros((len(f),8),dtype=np.float32)
extra_names=['own_ge_1min','own_ge_10min','own_ge_1h','own_cart','own_ord','own_recent_exact','own_queryless_exact','own_linked_exact']
counts=Counter(); n_hist=Counter(); structure=[]; raw_checks=Counter(); query_lists=defaultdict(list)
query_overlap=[]; raw_records=[]
say(f'Loaded {len(f):,} validation candidates / {len(size):,} searches; {len(items):,} distinct products')
with (ROOT/'data/user_behavior_data.txt').open(encoding='utf-8') as h:
    header=next(h)
    for row,line in enumerate(h):
        if row not in train_ids and row not in val_ids: continue
        fields=line.rstrip('\r\n').split('\t')
        q,c,lab,hq,hp,ht,hg=fields
        c=c.split('_'); lab=np.array(lab.split('_'),float).astype(np.int8)
        hp=hp.split('_'); ht=np.array(ht.split('_')); hq=hq.split('_'); gaps=np.array(hg.split('_'),dtype=np.int64)
        counts['records']+=1; counts['candidates']+=len(c)
        counts['duplicate_candidate_records']+=len(set(c))!=len(c)
        counts['candidate_label_mismatch']+=len(c)!=len(lab)
        counts['history_alignment_mismatch']+=not(len(hp)==len(ht)==len(hq)==len(gaps)-1)
        counts['negative_gap_records']+=bool((gaps<0).any())
        counts['nonzero_first_gap']+=gaps[0]!=0
        age=np.cumsum(gaps[::-1])[::-1][1:]
        npos=int((lab>0).sum()); nneg=len(lab)-npos
        structure.append((row, 'train' if row in train_ids else 'validation',len(c),npos,nneg,int(lab.max()),len(hp)))
        counts['all_positive_prefix']+=bool(np.all(lab[:npos]>0) and np.all(lab[npos:]==0))
        counts['all_labels_descending']+=bool(np.all(np.diff(lab)<=0))
        if row in train_ids:
            fold=int(np.random.default_rng([4222,7,row]).integers(5))
            fold_tot[fold]+= [npos,len(c)]
            for w,l in zip(c,lab):
                j=lookup.get(w)
                if j is not None: exp[fold,j]+=1; eng[fold,j]+=l>0
            idx=np.array([lookup.get(w,-1) for w in hp]); valid=idx>=0
            vi=idx[valid]; va=age[valid]; vt=ht[valid]
            np.add.at(hist[:,0],vi,1); np.add.at(hist[:,1],np.unique(vi),1)
            for k,cut in [(2,86400),(3,172800),(4,604800)]: np.add.at(hist[:,k],vi[va>cut],1)
            for k,typ in [(5,'CLICK'),(6,'CART'),(7,'ORD'),(8,'FLW')]: np.add.at(hist[:,k],vi[vt==typ],1)
            np.add.at(hist[:,9],np.unique(vi[va>86400]),1)
            for typ in ['CLICK','CART','ORD','FLW']:
                a=age[ht==typ]
                n_hist[typ]+=len(a); n_hist[typ+'_gt1d']+=int((a>86400).sum()); n_hist[typ+'_gt7d']+=int((a>604800).sum())
        else:
            a,b=loc[row]; order=np.argsort(np.array(c,dtype=np.int64)); cs=np.array(c)[order]
            raw_checks['candidate_id_mismatches']+=int(np.sum(np.array(cs,dtype=np.int64)!=f.wid.to_numpy()[a:b]))
            raw_checks['label_mismatches']+=int(np.sum(lab[order]!=y[a:b]))
            hc=Counter(hp)
            raw_checks['own_exact_count_mismatches']+=int(np.sum(np.array([hc[w] for w in cs])!=f.pooled_product.to_numpy()[a:b]))
            subsets=[age>=60,age>=600,age>=3600,ht=='CART',ht=='ORD',age<600,np.array(hq)=='-1',np.array(hq)!='-1']
            for k,mask in enumerate(subsets):
                ctr=Counter(w for w,ok in zip(hp,mask) if ok)
                own_extra[a:b,k]=[ctr[w] for w in cs]
            cset=set(c)
            for prev in query_lists[q][:3]: query_overlap.append(len(cset&prev)/len(cset|prev))
            if len(query_lists[q])<3: query_lists[q].append(cset)
            raw_records.append((row,q,len(hp),float((np.array(hq)=='-1').mean())))
        if counts['records']%25000==0: say(f"Scanned {counts['records']:,} non-test records")
say('Raw scan complete')
structure=pd.DataFrame(structure,columns=['session_id','split','n','positives','zeros','max_label','history_len'])
structure.to_csv(OUT/'candidate_structure.csv',index=False)
save('raw_integrity',dict(counts)); save('raw_feature_checks',dict(raw_checks)); save('training_history_counts',dict(n_hist))
save('repeated_query_overlap',{'pairs':len(query_overlap),'mean':np.mean(query_overlap),'median':np.median(query_overlap),'q25':np.quantile(query_overlap,.25),'q75':np.quantile(query_overlap,.75),'validation_unique_queries':len(query_lists)})
np.savez_compressed(OUT/'recomputed_counts.npz',items=items,hist=hist,exp=exp,eng=eng,fold_tot=fold_tot,own_extra=own_extra)
pd.DataFrame(raw_records,columns=['session_id','query','history_len','queryless_fraction']).to_parquet(OUT/'validation_queries.parquet',index=False)
save('count_definitions',{'hist_columns':names,'own_extra_columns':extra_names})
totalexp=exp.sum(axis=0); totaleng=eng.sum(axis=0)
vf=np.array([int(np.random.default_rng([4222,7,int(s)]).integers(5)) for s in search_ids])[sg]
ooexp=totalexp[fi]-exp[vf,fi]; ooeng=totaleng[fi]-eng[vf,fi]
prior=(fold_tot[:,0].sum()-fold_tot[vf,0])/(fold_tot[:,1].sum()-fold_tot[vf,1])
rate=(ooeng+20*prior)/(ooexp+20)
save('population_feature_checks',{'exposure_mismatches':int(np.sum(totalexp[fi]!=f.pop_exposures)), 'history_mismatches':int(np.sum(hist[fi,0]!=f.pop_history)), 'rate_mismatches_at_float32':int(np.sum(rate.astype(np.float32)!=f.pop_engaged_rate)), 'rate_max_abs_error':float(np.max(np.abs(rate-f.pop_engaged_rate)))})

# Independent vectorized exact expectation under uniformly random ties.
rank=np.arange(len(f))-np.repeat(starts,size)
discount=np.where(rank<10,1/np.log2(rank+2),0.)
yo=y[np.lexsort((-y,sid))]
ideal=np.bincount(sg,weights=yo*discount,minlength=len(size))
eligible=ideal>0
def ndcg(score):
    score=np.nan_to_num(np.asarray(score,dtype=float),nan=0.)
    order=np.lexsort((-score,sid)); ss=score[order]; yy=y[order]
    blocks=np.r_[0,np.flatnonzero((np.diff(ss)!=0)|(np.diff(sid[order])!=0))+1]
    blen=np.diff(np.r_[blocks,len(f)])
    gains=np.add.reduceat(yy,blocks)/blen
    discs=np.add.reduceat(discount,blocks)
    dcg=np.bincount(sg[blocks],weights=gains*discs,minlength=len(size))
    return np.divide(dcg,ideal,out=np.full(len(size),np.nan),where=eligible)
def interval(x, repeats=1000):
    x=np.asarray(x); x=x[np.isfinite(x)]; rng=np.random.default_rng(4222)
    if not len(x): return [None,None,None]
    means=np.array([x[rng.integers(0,len(x),len(x))].mean() for _ in range(repeats)])
    return [float(x.mean()),*map(float,np.quantile(means,[.025,.975]))]
validate_evaluation(); rng=np.random.default_rng(4222); checks=[]
for _ in range(30):
    n=int(rng.integers(2,7)); s=rng.integers(0,3,n); lab=rng.integers(0,4,n)
    if not lab.any(): lab[0]=1
    # Enumerate every candidate ordering consistent with descending score.
    allowed=[p for p in itertools.permutations(range(n)) if np.all(np.diff(s[list(p)])<=0)]
    d=1/np.log2(np.arange(2,n+2)); den=np.dot(np.sort(lab)[::-1],d)
    brute=np.mean([np.dot(lab[list(p)],d)/den for p in allowed])
    checks.append(abs(brute-expected_metrics(s,lab)[0]))
save('metric_bruteforce',{'cases':len(checks),'max_abs_error':max(checks)})
scores={'random':np.zeros(len(f)), 'exposures':f.pop_exposures, 'history_events':f.pop_history, 'engagement_rate':f.pop_engaged_rate,
        'jaccard':f.title_jaccard, 'bm25':f.bm25, 'own_exact':f.pooled_product, 'shop_other':f.pooled_shop_other,
        'brand_other':f.pooled_brand_other,'recent_click_exact':f.click_0_7_product}
scores.update({name:hist[fi,k] for k,name in enumerate(names) if k!=0})
scores.update({name:own_extra[:,k] for k,name in enumerate(extra_names) if k<3})
per={}; summary=[]
for name,s in scores.items():
    m=ndcg(s); per[name]=m
    summary.append([name,int(eligible.sum()),*interval(m)])
    say(f'{name}: NDCG {np.nanmean(m):.6f}')
    if name in ['random','history_events','own_exact']:
        sample=rng.choice(np.flatnonzero(eligible),100,replace=False)
        err=max(abs(expected_metrics(np.nan_to_num(np.asarray(s)[starts[j]:ends[j]],nan=0.),y[starts[j]:ends[j]])[0]-m[j]) for j in sample)
        assert err<1e-12,(name,err)
pd.DataFrame(summary,columns=['signal','searches','ndcg','ci_low','ci_high']).to_csv(OUT/'ranking_results.csv',index=False)
per=pd.DataFrame(per,index=search_ids); per.index.name='session_id'; per.to_parquet(OUT/'per_search.parquet')
comparisons=[]
for a,b in [('age_gt_1d','history_events'),('age_gt_2d','history_events'),('age_gt_7d','history_events'),('users','history_events'),('users_gt_1d','history_events'),('own_ge_1min','own_exact'),('own_ge_10min','own_exact'),('own_ge_1h','own_exact')]:
    comparisons.append([a,b,*interval(per[a]-per[b])])
pd.DataFrame(comparisons,columns=['a','b','delta','ci_low','ci_high']).to_csv(OUT/'paired_differences.csv',index=False)

# Structural/coverage and observed-label diagnostics, descriptive rather than causal.
f['hist_users']=hist[fi,1]; f['hist_old1d']=hist[fi,2]
for k,name in enumerate(extra_names): f[name]=own_extra[:,k]
f['has_own_exact']=f.pooled_product>0
f['seen_train_candidates']=f.pop_exposures>0; f['seen_train_history']=f.pop_history>0
f['pop_bin']=pd.cut(f.pop_history,[-1,0,1,3,10,30,100,300,1000,np.inf],right=True).astype(str)
f['length_bin']=pd.cut(f.n_candidates,[0,30,60,100,150,200]).astype(str)
desc=f.assign(engaged=f.label>0, purchased=f.label==3, lex=f.title_jaccard.fillna(0)>0)
for group in ['label','pop_bin','length_bin','in_catalogue','seen_train_candidates','seen_train_history']:
    d=desc.groupby(group,observed=True).agg(candidates=('label','size'),engagement_rate=('engaged','mean'),purchase_rate=('purchased','mean'),own_exact=('has_own_exact','mean'),pop_history_mean=('pop_history','mean'),lexical_overlap=('lex','mean'),catalogue_coverage=('in_catalogue','mean'))
    d.to_csv(OUT/f'profile_{group}.csv')
desc.groupby(['seen_train_candidates','seen_train_history'],observed=True).agg(candidates=('label','size'),engagement_rate=('engaged','mean')).to_csv(OUT/'coldness_cross.csv')
desc.assign(prior_cart=desc.own_cart>0,prior_ord=desc.own_ord>0,recent_exact=desc.own_recent_exact>0,queryless_exact=desc.own_queryless_exact>0,linked_exact=desc.own_linked_exact>0).groupby('label').agg(candidates=('label','size'),prior_cart=('prior_cart','mean'),prior_ord=('prior_ord','mean'),recent_exact=('recent_exact','mean'),queryless_exact=('queryless_exact','mean'),linked_exact=('linked_exact','mean')).to_csv(OUT/'prior_behavior_by_label.csv')
desc.groupby(['pop_bin','has_own_exact'],observed=True).agg(candidates=('label','size'),engagement_rate=('engaged','mean')).to_csv(OUT/'own_match_by_popularity.csv')
search=f.iloc[starts].set_index('session_id')
search['any_positive_exact']=np.bincount(sg,weights=((f.label>0)&f.has_own_exact).astype(int))>0
search['positive_count']=np.bincount(sg,weights=(f.label>0).astype(int)).astype(int)
segment=[]
for col in ['length_bin','any_positive_exact']:
    for val,ix in search.groupby(col,observed=True).groups.items():
        p=per.loc[ix].dropna()
        for name in ['history_events','age_gt_1d','users','own_exact','shop_other','jaccard']:
            segment.append([col,str(val),len(p),name,float(p[name].mean()),float(p['random'].mean()),*interval(p[name]-p['random'])])
pd.DataFrame(segment,columns=['segment','value','searches','signal','ndcg','random','gain','gain_low','gain_high']).to_csv(OUT/'segment_results.csv',index=False)
# Save narrow candidate data for inexpensive follow-up comparisons, avoiding the test set.
desc[['session_id','wid','label','pop_history','pop_exposures','hist_users','hist_old1d','pooled_product','pooled_shop_other','pooled_brand_other','title_jaccard','bm25','in_catalogue','n_candidates','hist_len','actions_last_7d','days_since_last_action','own_cart','own_ord','own_recent_exact','days_since_product','pop_bin']].to_parquet(OUT/'validation_audit.parquet',index=False)
save('run_metadata',{'elapsed_seconds':time.time()-t0,'validation_records':len(size),'evaluated_records':int(eligible.sum()),'training_records':len(train_ids),'test_records_read':0,'candidate_count':len(f),'seed':4222,'bootstrap_repeats':1000})
say('Audit complete; outputs saved')


## 2. Conditional pairs, candidate structure, E11 result reconciliation and ablation redundancy


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json, time
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'scripts/evaluation.py').exists())
OUT=ROOT/'outputs/data_generation_audit'; t0=time.time()
def save(name,obj): (OUT/f'{name}.json').write_text(json.dumps(obj,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)))
def ci(x):
    x=np.asarray(x); x=x[np.isfinite(x)]; r=np.random.default_rng(4222)
    m=np.array([x[r.integers(len(x),size=len(x))].mean() for _ in range(1000)])
    return [float(x.mean()),*map(float,np.quantile(m,[.025,.975]))]
f=pd.read_parquet(OUT/'validation_audit.parquet'); p=pd.read_parquet(OUT/'per_search.parquet')
s=pd.read_csv(OUT/'candidate_structure.csv')
save('structure_summary',{'records':len(s),'candidate_count_quantiles':s.n.quantile([0,.1,.25,.5,.75,.9,1]).to_dict(),
    'top_lengths':s.n.value_counts().head(15).to_dict(),'at_200':int((s.n==200).sum()),
    'zero_positive':int((s.positives==0).sum()),'four_negatives_per_positive':int(((s.zeros==4*s.positives)&(s.positives>0)).sum()),
    'spearman_length_positives':float(s.n.corr(s.positives,method='spearman'))})
s.groupby('positives').agg(records=('n','size'),mean_candidates=('n','mean'),median_candidates=('n','median'),min_candidates=('n','min'),max_candidates=('n','max')).to_csv(OUT/'length_by_positive_count.csv')

# Search-level pairwise discrimination, binary engagement versus zero; half credit for ties.
# Compare within the SAME search and identical or narrow population count bins.
features=['pop_history','pooled_product','pooled_shop_other','pooled_brand_other','title_jaccard']
sid=f.session_id.to_numpy(); starts=np.r_[0,np.flatnonzero(np.diff(sid))+1]; ends=np.r_[starts[1:],len(f)]
arr=f[features].fillna(0).to_numpy(); labels=f.label.to_numpy(); counts=f.pop_history.to_numpy()
popbin=np.floor(np.log2(counts+1)).astype(int); lexical=f.title_jaccard.fillna(0).to_numpy()>0
pairrows=[]
for a,b in zip(starts,ends):
    pos=np.flatnonzero(labels[a:b]>0)+a; neg=np.flatnonzero(labels[a:b]==0)+a
    if not len(pos) or not len(neg): continue
    delta=arr[pos,None,:]-arr[None,neg,:]
    concordance=(delta>0)+.5*(delta==0)
    masks={'all':np.ones((len(pos),len(neg)),bool),
           'same_pop_bin':popbin[pos,None]==popbin[None,neg],
           'same_pop_count':counts[pos,None]==counts[None,neg],
           'same_pop_bin_and_lex_overlap':(popbin[pos,None]==popbin[None,neg])&(lexical[pos,None]==lexical[None,neg])}
    for rule,mask in masks.items():
        if mask.any():
            vals=concordance[mask].mean(axis=0)
            varied=(delta[mask]!=0).mean(axis=0)
            pairrows.append([sid[a],rule,int(mask.sum()),*vals,*varied])
pairs=pd.DataFrame(pairrows,columns=['session_id','restriction','pairs',*features,*[v+'_variation' for v in features]])
pairs.to_parquet(OUT/'paired_candidate_diagnostics.parquet',index=False)
summary=[]
for rule,g in pairs.groupby('restriction'):
    for feat in features:
        summary.append([rule,feat,len(g),int(g.pairs.sum()),*ci(g[feat]),float(g[feat+'_variation'].mean())])
pd.DataFrame(summary,columns=['restriction','signal','searches','pairs','concordance','low','high','fraction_pairs_with_signal_difference']).to_csv(OUT/'paired_candidate_summary.csv',index=False)

# Exact-user signal after stratification; uncertainty at whole-search level.
# E11 numbers verified from persisted per-search outputs, not rerunning neural training.
lad=pd.read_parquet(ROOT/'data/processed/team3/ladder_per_search.parquet')
common=p.loc[lad.index]
save('e11_integrity',{'searches':len(lad),'same_eligible_searches':set(lad.index)==set(p.dropna().index),
    'max_random_error':float((lad['random order']-common.random).abs().max()),
    'max_popularity_error':float((lad['history popularity (E09)']-common.history_events).abs().max()),
    'r5_minus_popularity':ci(lad.iloc[:,-1]-common.history_events)})
reg=pd.read_csv(ROOT/'docs/experiment_results.csv'); reg=reg[reg.experiment.eq('E11')]
save('e11_registry_check',{'max_mean_error_vs_rounded_registry':float(np.max(np.abs(lad.iloc[:,2:].mean().to_numpy()-reg.ndcg_at_10.to_numpy())))})

# Query-cluster bootstrap, preserving search-weighted point estimate.
qs=pd.read_parquet(OUT/'validation_queries.parquet').set_index('session_id')
cluster=[]
for a,b in [('age_gt_1d','history_events'),('users','history_events'),('own_ge_10min','own_exact')]:
    d=(p[a]-p[b]).dropna(); g=pd.DataFrame({'d':d,'q':qs.loc[d.index,'query']}).groupby('q').d.agg(['sum','count'])
    sm=g['sum'].to_numpy(); n=g['count'].to_numpy(); r=np.random.default_rng(4222)
    vals=[]
    for _ in range(1000):
        ix=r.integers(0,len(g),len(g)); vals.append(sm[ix].sum()/n[ix].sum())
    cluster.append([a,b,len(g),float(d.mean()),*map(float,np.quantile(vals,[.025,.975]))])
pd.DataFrame(cluster,columns=['a','b','query_clusters','delta','ci_low','ci_high']).to_csv(OUT/'query_cluster_differences.csv',index=False)

# Multiplicity/context is not identified by drop-one-group ablations.
full=pd.read_parquet(ROOT/'data/processed/features_v2_validation.parquet')
identities=[]
for match in ['product','shop_other','brand_other']:
    pooled=full['pooled_'+match]
    bytype=full[[f'type_{t}_{match}' for t in ['click','cart','ord','flw']]].sum(axis=1,min_count=4)
    source=full['linked_'+match]+full['qless_'+match]
    for kind,v in [('type_sum',bytype),('source_sum',source)]:
        same=np.isclose(pooled,v,equal_nan=True)
        identities.append([match,kind,int((~same).sum())])
pd.DataFrame(identities,columns=['match','identity','mismatches']).to_csv(OUT/'ablation_redundancy.csv',index=False)
search=full.drop_duplicates('session_id')
save('context_proxy',{'searches':len(search),
    'recent_activity_exactly_recoverable_from_C1_counts':bool(np.all(search.actions_last_7d==search.n_click_0_7+search.n_cart_0_7+search.n_ord_0_7)),
    'recent_activity_proxy_mismatches_due_to_follow_or_stragglers':int(np.sum(search.actions_last_7d!=search.n_click_0_7+search.n_cart_0_7+search.n_ord_0_7)),
    'history_over_200':int((search.hist_len>200).sum())})
save('followup_runtime',{'elapsed_seconds':time.time()-t0})
print(pd.DataFrame(summary,columns=['restriction','signal','searches','pairs','concordance','low','high','variation']).to_string(index=False))
print('Follow-up complete',time.time()-t0)


## 3. Independent catalogue reconstruction and historical missingness sample


In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import json,time
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'scripts/evaluation.py').exists())
OUT=ROOT/'outputs/data_generation_audit'; t=time.time(); rng=np.random.default_rng(4222)
split=pd.read_csv(ROOT/'data/processed/split_v1.csv')
sample=set(rng.choice(split.loc[split.split.eq('validation'),'session_id'],300,replace=False).tolist())
records=[]; needed=set()
with (ROOT/'data/user_behavior_data.txt').open(encoding='utf-8') as h:
    next(h)
    for sid,line in enumerate(h):
        if sid not in sample: continue
        q,c,y,hq,hp,ht,hg=line.rstrip('\r\n').split('\t')
        c=c.split('_'); hp=hp.split('_'); ht=ht.split('_'); hq=hq.split('_')
        g=np.array(hg.split('_'),np.int64); age=np.cumsum(g[::-1])[::-1][1:]/86400
        records.append((sid,q,c,hp,ht,hq,age)); needed.update(c);needed.update(hp)
print('Sample read',len(needed),'metadata items',flush=True)
meta={}; background=[]; offset=int(rng.integers(1000))
with (ROOT/'data/product_meta_data.txt').open(encoding='utf-8') as h:
    next(h)
    for i,line in enumerate(h):
        w=line.split('\t',1)[0]
        take=i%1000==offset
        if w not in needed and not take:continue
        v=line.rstrip('\r\n').split('\t')
        if w in needed:meta[w]=v
        if take:background.append(v[1])
print('Catalogue scanned',i+1,'rows in',time.time()-t,'s',flush=True)
def terms(q): return set(q.split('\x18')) if q not in ('','-1') else set()
def key(w,col):
    if w not in meta:return None
    x=meta[w][col];return x if x not in ('','-1') else None
def l3(w):
    x=tuple(key(w,i) for i in [4,6,8]);return x if None not in x else None
f=pd.read_parquet(ROOT/'data/processed/features_v2_validation.parquet').set_index(['session_id','wid'])
checks=Counter(); checked=Counter(); miss=[]; lex=[]
def compare(name,got,expected):
    checked[name]+=1
    if not np.isclose(got,expected,atol=1e-7,rtol=1e-6,equal_nan=True):checks[name]+=1
for sid,q,c,hp,ht,hq,age in records:
    qset=terms(q); hc=Counter(hp)
    byshop=Counter(key(w,12) for w in hp if key(w,12) is not None)
    bybrand=Counter(key(w,2) for w in hp if key(w,2) is not None)
    bycat=Counter(l3(w) for w in hp if l3(w) is not None)
    for w in c:
        row=f.loc[(sid,int(w))]
        vals={'in_catalogue':int(w in meta),'pooled_product':hc[w],
              'pooled_shop_other':byshop[key(w,12)]-hc[w] if key(w,12) is not None else np.nan,
              'pooled_brand_other':bybrand[key(w,2)]-hc[w] if key(w,2) is not None else np.nan,
              'pooled_L3':bycat[l3(w)] if l3(w) is not None else np.nan}
        ts=terms(meta[w][1]) if w in meta else set()
        vals['title_jaccard']=len(qset&ts)/len(qset|ts) if ts else np.nan
        for name,v in vals.items():compare(name,v,row[name])
        if ts:lex.append(('candidate',int(bool(qset&ts))))
    for title in rng.choice(background,100,replace=False):
        ts=terms(title)
        if ts:lex.append(('catalogue_background',int(bool(qset&ts))))
    for w,typ,a in zip(hp,ht,age):miss.append((sid,typ,'0-7' if a<7 else '7-90' if a<90 else '90+',w not in meta))
miss=pd.DataFrame(miss,columns=['session_id','type','age','missing'])
miss.groupby(['type','age']).agg(events=('missing','size'),missing_fraction=('missing','mean')).to_csv(OUT/'sample_history_metadata_coverage.csv')
cirows=[]
for (typ,age),part in miss.groupby(['type','age']):
    g=part.groupby('session_id').missing.agg(['sum','count']).reindex(sorted(sample),fill_value=0)
    r=np.random.default_rng(4222); sums=g['sum'].to_numpy(); nn=g['count'].to_numpy(); vals=[]
    for _ in range(1000):
        ix=r.integers(0,len(sample),len(sample)); den=nn[ix].sum()
        if den: vals.append(sums[ix].sum()/den)
    cirows.append([typ,age,len(part),part.session_id.nunique(),float(part.missing.mean()),*map(float,np.quantile(vals,[.025,.975]))])
pd.DataFrame(cirows,columns=['type','age','events','contributing_searches','missing_rate','ci_low','ci_high']).to_csv(OUT/'sample_history_metadata_intervals.csv',index=False)
miss.to_parquet(OUT/'sample_history_metadata_events.parquet',index=False)
pd.DataFrame(lex,columns=['source','overlap']).groupby('source').agg(pairs=('overlap','size'),overlap_fraction=('overlap','mean')).to_csv(OUT/'lexical_background_comparison.csv')
(OUT/'metadata_reconstruction.json').write_text(json.dumps({'sample_searches':len(sample),'catalogue_rows_scanned':i+1,'candidate_checks_per_feature':dict(checked),'mismatches':{k:int(checks[k]) for k in checked},'runtime_seconds':time.time()-t},indent=2))
print('Metadata reconstruction',dict(checks),'runtime',time.time()-t)


## 4. E11 fitting coverage and sequence truncation


In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'scripts/evaluation.py').exists())
OUT=ROOT/'outputs/data_generation_audit'; D=ROOT/'data/processed/team3'
tr=np.load(D/'train_dev.npz'); va=np.load(D/'validation.npz'); info=json.loads((D/'meta.json').read_text())
rows=~tr['inner_val']; candidate_rows=np.repeat(rows,np.diff(tr['cand_offsets']))
ids=tr['cand_idx']; v=va['cand_idx']; warm_all=np.isin(v,np.unique(ids)); warm_fit=np.isin(v,np.unique(ids[candidate_rows]))
labels=tr['label'][candidate_rows]; counts=np.bincount(labels,minlength=4)
w=(1/np.maximum(counts,1));w/=w.sum()
f=pd.read_parquet(OUT/'validation_audit.parquet'); d=pd.read_csv(OUT/'candidate_structure.csv'); d=d[d.split=='validation'].set_index('session_id').loc[va['session_id']]
length=np.diff(va['hist_full_offsets'])
out={'candidate_occurrences':len(v),'warm_to_40k_dev_lists':float(warm_all.mean()),'warm_to_36k_fit_lists':float(warm_fit.mean()),
     'warm_to_full_133831_train_lists':float((f.pop_exposures>0).mean()),'warm_to_full_train_histories':float((f.pop_history>0).mean()),
     'fit_full_list_label_counts':counts.tolist(),'normalized_inverse_frequency_class_weights':w.tolist(),
     'validation_histories_capped_at_200':int((d.history_len>200).sum()),
     'saved_lengths_equal_min_raw_200':bool(np.array_equal(length,np.minimum(d.history_len,200))),
     'max_encoded_history':int(length.max())}
(OUT/'e11_coverage.json').write_text(json.dumps(out,indent=2));print(json.dumps(out,indent=2))


## 5. Exhaustive cutoff tests, full shared-evaluator comparison and result registration


In [ ]:
from pathlib import Path
import sys,json,itertools,time
import numpy as np,pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'scripts/evaluation.py').exists())
sys.path.insert(0,str(ROOT))
from scripts.evaluation import expected_metrics,evaluate,log_result
OUT=ROOT/'outputs/data_generation_audit'; t=time.time(); rng=np.random.default_rng(4222)
errors=[]
for _ in range(30):
    n=int(rng.integers(2,7)); s=rng.integers(0,3,n); y=rng.integers(0,4,n)
    if not y.any():y[0]=1
    orders=[p for p in itertools.permutations(range(n)) if np.all(np.diff(s[list(p)])<=0)]
    for k in [1,min(3,n),n]:
        d=1/np.log2(np.arange(2,k+2)); ideal=np.dot(np.sort(y)[::-1][:k],d)
        brute=np.mean([np.dot(y[list(p)][:k],d)/ideal for p in orders])
        errors.append(abs(brute-expected_metrics(s,y,k)[0]))
assert max(errors)<1e-12
(OUT/'metric_bruteforce_cutoffs.json').write_text(json.dumps({'cases':len(errors),'cutoffs':'1, min(3,n), n','max_abs_error':max(errors)},indent=2))
f=pd.read_parquet(OUT/'validation_audit.parquet'); per=pd.read_parquet(OUT/'per_search.parquet')
z=np.load(OUT/'recomputed_counts.npz'); idx=np.searchsorted(z['items'],f.wid.to_numpy()); hist=z['hist']; own=z['own_extra']
signals={'users':hist[idx,1],'age_gt_1d':hist[idx,2],'age_gt_2d':hist[idx,3],'age_gt_7d':hist[idx,4],
         'click':hist[idx,5],'cart':hist[idx,6],'ord':hist[idx,7],'flw':hist[idx,8],'users_gt_1d':hist[idx,9],
         'own_ge_1min':own[:,0],'own_ge_10min':own[:,1],'own_ge_1h':own[:,2]}
sizes=f.groupby('session_id').size(); reg=pd.read_csv(ROOT/'docs/experiment_results.csv')
existing=set(reg.loc[reg.experiment.eq('E12'),'model']); verification=[]
for name,score in signals.items():
    frame=f[['session_id','label']].copy();frame['score']=score
    actual=evaluate(frame,expected_sizes=sizes)
    err=float((actual.ndcg-per.loc[actual.index,name]).abs().max()); assert err<1e-12,(name,err)
    verification.append({'signal':name,'searches':len(actual),'max_ndcg_difference':err})
    model='audit: '+name
    if model not in existing:
        log_result('E12',model,'validation',actual,notes='Fixed untrained signal; full lists; independent reconstruction; audit notebook; donor-relative historical lag where applicable')
    print(name,'verified and registered',flush=True)
(OUT/'full_metric_crosscheck.json').write_text(json.dumps({'signals':verification,'seconds':time.time()-t},indent=2))


## Recorded result snapshot

These tables were loaded from the generated audit outputs after the scripts completed. They are a result record, not an additional experiment. Re-run the code cells above to regenerate the underlying files.

```
            signal  searches     ndcg   ci_low  ci_high
            random     19772 0.084481 0.083040 0.086023
         exposures     19772 0.315018 0.310660 0.319656
    history_events     19772 0.340944 0.336386 0.345528
   engagement_rate     19772 0.244990 0.240942 0.249361
           jaccard     19772 0.097307 0.094406 0.100226
              bm25     19772 0.092979 0.090271 0.095770
         own_exact     19772 0.181985 0.178337 0.185798
        shop_other     19772 0.135628 0.132907 0.138363
       brand_other     19772 0.098456 0.096323 0.100700
recent_click_exact     19772 0.155351 0.151974 0.158911
             users     19772 0.340800 0.336209 0.345304
         age_gt_1d     19772 0.336950 0.332435 0.341428
         age_gt_2d     19772 0.334582 0.330022 0.339117
         age_gt_7d     19772 0.324162 0.319424 0.328569
             click     19772 0.320768 0.316595 0.325602
              cart     19772 0.331093 0.326303 0.335934
               ord     19772 0.283861 0.279793 0.288249
               flw     19772 0.121005 0.118154 0.123613
       users_gt_1d     19772 0.336961 0.332271 0.341506
       own_ge_1min     19772 0.166067 0.162713 0.169627
      own_ge_10min     19772 0.146744 0.143454 0.150023
         own_ge_1h     19772 0.139098 0.135988 0.142102
```

Paired changes:

```
           a              b     delta    ci_low   ci_high
   age_gt_1d history_events -0.003995 -0.004924 -0.003139
   age_gt_2d history_events -0.006362 -0.007540 -0.005240
   age_gt_7d history_events -0.016783 -0.018677 -0.015030
       users history_events -0.000144 -0.001412  0.001104
 users_gt_1d history_events -0.003983 -0.005404 -0.002599
 own_ge_1min      own_exact -0.015917 -0.017484 -0.014367
own_ge_10min      own_exact -0.035240 -0.037546 -0.033056
   own_ge_1h      own_exact -0.042887 -0.045438 -0.040454
```
